### DIMUSER 

In [0]:
%load_ext autoreload
%autoreload 2

from pyspark.sql.functions import *
from pyspark.sql.types import *

import os
import sys

project_path = os.path.abspath(
    os.path.join(os.getcwd(), "..", "..")
)

if project_path not in sys.path:
    sys.path.insert(0, project_path)

from utils.transformations import reusable

### AUTOLOADER 

In [0]:
user_source_path = (
    "abfss://bronze@spotifystorage2712.dfs.core.windows.net/DimUser"
)

user_checkpoint_path = (
    "abfss://silver@spotifystorage2712.dfs.core.windows.net/"
    "DimUser/checkpoint"
)

user_data_path = (
    "abfss://silver@spotifystorage2712.dfs.core.windows.net/"
    "DimUser/data"
)

df_user_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", user_checkpoint_path)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.includeExistingFiles", "true")
        .load(user_source_path)
)

In [0]:
df_user = spark.sql("""
    SELECT *
    FROM read_files(
        'abfss://bronze@spotifystorage2712.dfs.core.windows.net/DimUser',
        format => 'parquet'
    )
""")

display(df_user)

### Transformations 


In [0]:
#uppercase
df_user_stream = df_user_stream.withColumn(
    "user_name",
    upper(col("user_name"))
)

df_user = df_user.withColumn(
    "user_name",
    upper(col("user_name"))
)

display(df_user)

In [0]:
#drop columns and duplicates
df_user_obj = reusable()

# Actual streaming DataFrame
df_user_stream = df_user_obj.dropColumns(
    df_user_stream,
    ["_rescued_data"]
)

df_user_stream = df_user_obj.dropDuplicates(
    df_user_stream,
    ["user_id"]
)

# Batch DataFrame used for display
df_user = df_user_obj.dropColumns(
    df_user,
    ["_rescued_data"]
)

df_user = df_user_obj.dropDuplicates(
    df_user,
    ["user_id"]
)

display(df_user)

In [0]:
#write
df_user_stream.writeStream.format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        user_checkpoint_path
    ) \
    .trigger(once=True) \
    .option(
        "path",
        user_data_path
    ) \
    .toTable("databricks_spotify.silver.DimUser")

In [0]:
display(
    spark.table("databricks_spotify.silver.DimUser")
)

### DimArtist 


In [0]:
#Autoloader

artist_source_path = (
    "abfss://bronze@spotifystorage2712.dfs.core.windows.net/DimArtist"
)

artist_checkpoint_path = (
    "abfss://silver@spotifystorage2712.dfs.core.windows.net/"
    "DimArtist/checkpoint"
)

artist_data_path = (
    "abfss://silver@spotifystorage2712.dfs.core.windows.net/"
    "DimArtist/data"
)

df_artist_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", artist_checkpoint_path)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.includeExistingFiles", "true")
        .load(artist_source_path)
)

In [0]:
df_artist = spark.sql("""
    SELECT *
    FROM read_files(
        'abfss://bronze@spotifystorage2712.dfs.core.windows.net/DimArtist',
        format => 'parquet'
    )
""")

display(df_artist)

In [0]:
df_artist_obj = reusable()

# Actual streaming DataFrame
df_artist_stream = df_artist_obj.dropColumns(
    df_artist_stream,
    ["_rescued_data"]
)

df_artist_stream = df_artist_obj.dropDuplicates(
    df_artist_stream,
    ["artist_id"]
)

# Batch DataFrame used for display
df_artist = df_artist_obj.dropColumns(
    df_artist,
    ["_rescued_data"]
)

df_artist = df_artist_obj.dropDuplicates(
    df_artist,
    ["artist_id"]
)

display(df_artist)

In [0]:
df_artist_stream.writeStream.format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "abfss://silver@spotifystorage2712.dfs.core.windows.net/DimArtist/checkpoint"
    ) \
    .trigger(once=True) \
    .option(
        "path",
        "abfss://silver@spotifystorage2712.dfs.core.windows.net/DimArtist/data"
    ) \
    .toTable("databricks_spotify.silver.DimArtist")

In [0]:
display(
    spark.table("databricks_spotify.silver.DimArtist")
)

### DIMTRACK


In [0]:
track_source_path = (
    "abfss://bronze@spotifystorage2712.dfs.core.windows.net/DimTrack"
)

track_checkpoint_path = (
    "abfss://silver@spotifystorage2712.dfs.core.windows.net/"
    "DimTrack/checkpoint"
)

track_data_path = (
    "abfss://silver@spotifystorage2712.dfs.core.windows.net/"
    "DimTrack/data"
)

df_track_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", track_checkpoint_path)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.includeExistingFiles", "true")
        .load(track_source_path)
)

In [0]:
df_track = spark.sql("""
    SELECT *
    FROM read_files(
        'abfss://bronze@spotifystorage2712.dfs.core.windows.net/DimTrack',
        format => 'parquet'
    )
""")

display(df_track)

In [0]:
df_track_stream = df_track_stream.withColumn(
    "durationFlag",
    when(col("duration_sec") < 150, "low")
        .when(col("duration_sec") < 300, "medium")
        .otherwise("high")
)

df_track_stream = df_track_stream.withColumn(
    "track_name",
    regexp_replace(col("track_name"), "-", " ")
)

df_track_stream = reusable().dropColumns(
    df_track_stream,
    ["_rescued_data"]
)

In [0]:
df_track_preview = spark.sql("""
    SELECT *
    FROM read_files(
        'abfss://bronze@spotifystorage2712.dfs.core.windows.net/DimTrack',
        format => 'parquet'
    )
""")

df_track_preview = df_track_preview.withColumn(
    "durationFlag",
    when(col("duration_sec") < 150, "low")
        .when(col("duration_sec") < 300, "medium")
        .otherwise("high")
)

df_track_preview = df_track_preview.withColumn(
    "track_name",
    regexp_replace(col("track_name"), "-", " ")
)

df_track_preview = reusable().dropColumns(
    df_track_preview,
    ["_rescued_data"]
)

display(df_track_preview)

In [0]:
track_query = (
    df_track_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            track_checkpoint_path
        )
        .trigger(once=True)
        .option(
            "path",
            track_data_path
        )
        .toTable(
            "databricks_spotify.silver.DimTrack"
        )
)

track_query.awaitTermination()

In [0]:
track_table = spark.table(
    "databricks_spotify.silver.DimTrack"
)

display(
    track_table.select(
        "track_id",
        "track_name",
        "duration_sec",
        "durationFlag"
    )
)

print("Total rows:", track_table.count())
print("_rescued_data exists:", "_rescued_data" in track_table.columns)

### DIMDATE

In [0]:
date_source_path = (
    "abfss://bronze@spotifystorage2712.dfs.core.windows.net/DimDate"
)

date_checkpoint_path = (
    "abfss://silver@spotifystorage2712.dfs.core.windows.net/"
    "DimDate/checkpoint"
)

date_data_path = (
    "abfss://silver@spotifystorage2712.dfs.core.windows.net/"
    "DimDate/data"
)

df_date_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", date_checkpoint_path)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.includeExistingFiles", "true")
        .load(date_source_path)
)

In [0]:
df_date_stream = reusable().dropColumns(
    df_date_stream,
    ["_rescued_data"]
)

In [0]:
date_query = (
    df_date_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            date_checkpoint_path
        )
        .trigger(once=True)
        .option(
            "path",
            date_data_path
        )
        .toTable(
            "databricks_spotify.silver.DimDate"
        )
)

date_query.awaitTermination()

In [0]:
date_table = spark.table(
    "databricks_spotify.silver.DimDate"
)

display(date_table)

print("Total rows:", date_table.count())
print("_rescued_data exists:", "_rescued_data" in date_table.columns)

### FACTSTREAM

In [0]:
fact_source_path = (
    "abfss://bronze@spotifystorage2712.dfs.core.windows.net/FactStream"
)

fact_checkpoint_path = (
    "abfss://silver@spotifystorage2712.dfs.core.windows.net/"
    "FactStream/checkpoint"
)

fact_data_path = (
    "abfss://silver@spotifystorage2712.dfs.core.windows.net/"
    "FactStream/data"
)

df_fact_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", fact_checkpoint_path)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.includeExistingFiles", "true")
        .load(fact_source_path)
)

In [0]:
df_fact = spark.sql("""
    SELECT *
    FROM read_files(
        'abfss://bronze@spotifystorage2712.dfs.core.windows.net/FactStream',
        format => 'parquet'
    )
""")

display(df_fact)

In [0]:
df_fact_stream = reusable().dropColumns(
    df_fact_stream,
    ["_rescued_data"]
)

In [0]:
fact_query = (
    df_fact_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            fact_checkpoint_path
        )
        .trigger(once=True)
        .option(
            "path",
            fact_data_path
        )
        .toTable(
            "databricks_spotify.silver.FactStream"
        )
)

fact_query.awaitTermination()

In [0]:
fact_table = spark.table(
    "databricks_spotify.silver.FactStream"
)

display(fact_table)

print("Total rows:", fact_table.count())
print("_rescued_data exists:", "_rescued_data" in fact_table.columns)